# Tutorial: Validazione Forecast Clima su Colab GPU

Questo notebook serve a eseguire la validazione ufficiale del forecast `temperatura + precipitazione` su GPU gratuita (`Google Colab`) usando il codice gia presente nel progetto.

Obiettivi:
- verificare che la runtime Colab abbia davvero `cuda` disponibile;
- eseguire il test città 2019 sul checkpoint `small`;
- eseguire il test aree non urbane;
- leggere subito i report prodotti senza cercare i file a mano.


## Prerequisiti

Prima di aprire questo notebook, prepara il subset minimo sul tuo Mac:

```bash
cd /Users/simonemercolino/Desktop/Università/Tesi_BioMap/TCBiomap/tesi_bioanalyst_repo
source scripts/activate_bioanalyst_model.sh
python scripts/prepare_colab_validation_subset.py --target-dir /percorso/del/tuo/staging_colab --clean
```

Poi copia la cartella risultante su Google Drive, ad esempio in:

- `/content/drive/MyDrive/biomap_validation_subset`


In [ ]:
# Step 1 - Clona il repository e installa le dipendenze.
# Se stai riaprendo una sessione Colab, puoi rieseguire questa cella in sicurezza.

!if [ ! -d /content/tesi_bioanalyst_repo ]; then git clone https://github.com/ShimOne420/tesi_bioanalyst_repo.git /content/tesi_bioanalyst_repo; fi
!mkdir -p /content/tesi_bioanalyst_repo/external
!if [ ! -d /content/tesi_bioanalyst_repo/external/bfm-model ]; then git clone https://github.com/BioDT/bfm-model.git /content/tesi_bioanalyst_repo/external/bfm-model; fi
%cd /content/tesi_bioanalyst_repo
!cd /content/tesi_bioanalyst_repo && pip install -r requirements.txt
!cd /content/tesi_bioanalyst_repo/external/bfm-model && pip install -e .


In [ ]:
# Step 2 - Monta Google Drive e imposta i path del subset minimo.
# Aggiorna DRIVE_ROOT solo se hai usato un percorso diverso su Drive.

from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/biomap_validation_subset')
os.environ['BIOCUBE_DIR'] = str(DRIVE_ROOT / 'biocube')
os.environ['BIOANALYST_MODEL_DIR'] = str(DRIVE_ROOT / 'models')
os.environ['PROJECT_OUTPUT_DIR'] = str(DRIVE_ROOT / 'outputs')

print('BIOCUBE_DIR =', os.environ['BIOCUBE_DIR'])
print('BIOANALYST_MODEL_DIR =', os.environ['BIOANALYST_MODEL_DIR'])
print('PROJECT_OUTPUT_DIR =', os.environ['PROJECT_OUTPUT_DIR'])


In [ ]:
# Step 3 - Verifica che Colab abbia davvero una GPU NVIDIA disponibile.
# Se `cuda` e False, questo non e un problema del progetto: va semplicemente riaperta una runtime GPU.

import torch

print('torch version =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
print('cuda device count =', torch.cuda.device_count())
if torch.cuda.is_available():
    print('cuda device name =', torch.cuda.get_device_name(0))


In [ ]:
# Step 4 - Test ufficiale città 2019 sul checkpoint small.
# Questo e il run principale da cui partire per decidere se il forecast clima e credibile.

!cd /content/tesi_bioanalyst_repo && python scripts/forecast_validate_climate.py --forecast-start 2019-01-01 --forecast-end 2019-12-01 --month-stride 1 --checkpoint small --device cuda


In [ ]:
# Step 5 - Test non urbano 2019.
# Questo aiuta a capire se il modello si comporta meglio fuori dai centri urbani.

!cd /content/tesi_bioanalyst_repo && python scripts/forecast_validate_climate.py --areas-json data/validation_non_urban_areas.json --forecast-start 2019-01-01 --forecast-end 2019-12-01 --month-stride 1 --checkpoint small --device cuda


In [ ]:
# Step 6 - Test storico piu leggero 2000-2020 con stride trimestrale.
# E consigliato solo dopo aver visto che i run 2019 finiscono correttamente.

!cd /content/tesi_bioanalyst_repo && python scripts/forecast_validate_climate.py --forecast-start 2000-03-01 --forecast-end 2020-12-01 --month-stride 3 --checkpoint small --device cuda


In [ ]:
# Step 7 - Ispeziona l'ultimo report con lo script di sintesi.
# Questo stampa migliori e peggiori citta/aree e salva un report Markdown leggibile.

!cd /content/tesi_bioanalyst_repo && python scripts/inspect_forecast_validation_report.py


In [ ]:
# Step 8 - Trova i file finali prodotti dal run piu recente.
# Da qui puoi aprire direttamente gli XLSX dal pannello file di Colab o da Google Drive.

!find "$PROJECT_OUTPUT_DIR" -name "forecast_validation_climate*.xlsx" | sort
!find "$PROJECT_OUTPUT_DIR" -name "forecast_validation_climate*.json" | sort
!find "$PROJECT_OUTPUT_DIR" -name "forecast_validation_climate_report.md" | sort


## Come interpretare il risultato

Il forecast clima va considerato `convincente` solo se, in media e non solo su un caso singolo:

- la temperatura passa sia la soglia percentuale in Kelvin sia la soglia assoluta in `°C`;
- la precipitazione passa sia la soglia `sMAPE` sia la soglia assoluta in `mm`;
- il `city_climate_pass_share` nel summary non resta vicino a zero.

Se il test 2019 fallisce gia qui, il passo successivo non e la UI: e il confronto tra `small` e `large` oppure un riallineamento piu stretto al preprocessing ufficiale.
